# 🎙️ Comparativa de clonación de voz en inglés en Google Colab

Este notebook genera **el mismo texto en inglés con 3 modelos diferentes**, usando una grabación de referencia de tu voz:

1. **F5-TTS** — zero-shot voice cloning.
2. **XTTS-v2 (Coqui)** — voice cloning multilingüe.
3. **OpenVoice V2** — clonación del timbre sobre una voz TTS base.

Al final podrás escuchar los tres audios uno detrás de otro y comparar cuál conserva mejor tu voz.

> **Uso responsable:** utiliza únicamente tu propia voz o una voz para la que tengas consentimiento explícito.

---

## Antes de empezar

En Colab ve a:

**Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**

Después ejecuta las celdas **en orden, de arriba abajo**.

### Grabación recomendada

Para que la clonación funcione bien:

- Habla **en inglés**.
- Usa entre **8 y 15 segundos**.
- Una sola persona hablando.
- Sin música ni ruido de fondo.
- Habla con tu tono natural.
- WAV, MP3, M4A u OGG sirven: el notebook lo convertirá a WAV mono automáticamente.

Ejemplo de grabación:

> *Welcome back! Today, we're talking about what's happening in the financial markets, because there is a lot to discuss. Stock prices are moving and investors are asking the same question: what happens next? So, let's take a look at the latest data and find out what we should be watching. Let's get started!*

## 1. Comprobar que Colab tiene GPU

Si aparece `Tesla T4`, `L4`, `A100` o similar, está bien.

Si aparece `CPU`, cambia el entorno de ejecución antes de continuar.

In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No hay GPU. En Colab: Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU"
    )

PyTorch: 2.11.0+cu130
CUDA disponible: True
GPU: Tesla T4


## 2. Instalar herramientas comunes

Esta celda instala `ffmpeg`, utilidades de audio y Whisper para transcribir automáticamente tu grabación.

Puede tardar un par de minutos.

In [2]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install -U soundfile librosa IPython openai-whisper

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 16.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 639.0/639.0 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 123.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.2/86.2 kB 9.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 re

## 3. Subir tu grabación

Ejecuta la celda y selecciona el archivo desde tu ordenador.

El notebook:
1. detecta el archivo;
2. lo convierte a WAV mono de 24 kHz;
3. lo guarda como `/content/reference.wav`.

**Solo sube un archivo de voz.**

In [33]:
# Clonar el repositorio FinTok
!git clone https://github.com/mirdbg/FinTok.git /content/FinTok

fatal: destination path '/content/FinTok' already exists and is not an empty directory.


In [34]:
# Actualizar
%cd /content/FinTok
!git pull
%cd /content

/content/FinTok
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 10 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 536.85 KiB | 2.51 MiB/s, done.
From https://github.com/mirdbg/FinTok
   373bc91..0edc7cc  main       -> origin/main
Updating 373bc91..0edc7cc
Fast-forward
 .../{01_EN_Miriam.wav => 01_EN_MIRIAM_LARGE_FAST.wav}  | Bin
 data/audio_samples/02_EN_MIRIAM_SHORT_FAST.wav         | Bin 0 -> 655438 bytes
 data/audio_samples/03_EN_MIRIAM_SHORT_SLOW.wav         | Bin 0 -> 917582 bytes
 3 files changed, 0 insertions(+), 0 deletions(-)
 rename data/audio_samples/{01_EN_Miriam.wav => 01_EN_MIRIAM_LARGE_FAST.wav} (100%)
 create mode 100644 data/audio_samples/02_EN_MIRIAM_SHORT_FAST.wav
 create mode 100644 data/audio_samples/03_EN_MIRIAM_SHORT_SLOW.wav
/content


In [52]:
from pathlib import Path
import subprocess

# Audio guardado dentro del repositorio FinTok
input_path = Path("/content/FinTok/data/audio_samples/02_EN_MIRIAM_SHORT_FAST.wav")

# Archivo normalizado que utilizarán los modelos
reference_path = Path("/content/reference.wav")

# Comprobar que existe
if not input_path.exists():
    raise FileNotFoundError(
        f"No encuentro el audio en: {input_path}\\n"
        "Comprueba que has clonado el repositorio FinTok en /content."
    )

# Convertir a WAV mono, 24 kHz
subprocess.run([
    "ffmpeg", "-y",
    "-i", str(input_path),
    "-ac", "1",
    "-ar", "24000",
    "-vn",
    str(reference_path)
], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("✓ Audio original:", input_path)
print("✓ Audio preparado:", reference_path)

✓ Audio original: /content/FinTok/data/audio_samples/02_EN_MIRIAM_SHORT_FAST.wav
✓ Audio preparado: /content/reference.wav


## 4. Escuchar la referencia

Comprueba que se oye limpia, sin música y sin silencios excesivos.

Si la grabación no es buena, vuelve a ejecutar la celda anterior y sube otra.

In [53]:
from IPython.display import Audio, display
display(Audio("/content/reference.wav"))

## 5. Transcribir automáticamente la grabación

F5-TTS funciona mejor cuando conoce el texto exacto pronunciado en el audio de referencia.

Usaremos Whisper para obtenerlo automáticamente. **Revisa el resultado.**  
Si Whisper se equivoca, corrige manualmente `REFERENCE_TEXT`.

In [45]:
import whisper

whisper_model = whisper.load_model("small.en")
result = whisper_model.transcribe("/content/reference.wav", language="en")
REFERENCE_TEXT = result["text"].strip()

print("TRANSCRIPCIÓN:")
print(REFERENCE_TEXT)

TRANSCRIPCIÓN:
Welcome back. Stock prices are moving and investors are asking the same question. What happens next? Let's get started.


### Opcional: corregir la transcripción

Solo modifica esta celda si la transcripción anterior no coincide exactamente con lo que dijiste.

Si está bien, puedes dejarla sin ejecutar.

In [47]:
# Descomenta y escribe aquí la transcripción correcta si hace falta:
REFERENCE_TEXT = """Welcome back. Stock prices are moving and investors are asking the same question. What happens next? Let's get started."""

## 6. Elegir el texto que quieres generar

Cambia `TARGET_TEXT` por cualquier texto en inglés.

Para la primera prueba conviene usar unas 2–4 frases. Una vez comprobado qué modelo te gusta más, puedes generar textos más largos.

In [48]:
TARGET_TEXT = """
Did you know that even though Apple’s internal controls are audited, the report explicitly states they might not always prevent or detect errors?
This is FinTok — your daily dose of financial knowledge.
""".strip()


print(TARGET_TEXT)

Did you know that even though Apple’s internal controls are audited, the report explicitly states they might not always prevent or detect errors?
This is FinTok — your daily dose of financial knowledge.


# MODELO 1 — F5-TTS

F5-TTS utiliza el audio y su transcripción como referencia para generar el nuevo texto manteniendo las características de la voz.

La primera ejecución descarga los pesos del modelo y será más lenta.

## 7. Instalar F5-TTS

In [15]:
!pip -q install -U f5-tts

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.3/107.3 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 77.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 8. Generar audio con F5-TTS

El resultado se guardará como `/content/output_f5.wav`.

In [54]:
import soundfile as sf
from f5_tts.api import F5TTS

f5 = F5TTS()

wav, sr, _ = f5.infer(
    ref_file="/content/reference.wav",
    ref_text=REFERENCE_TEXT,
    gen_text=TARGET_TEXT,
    speed=1,
)

sf.write("/content/output_f5.wav", wav, sr)
print("✓ F5-TTS terminado")

Download Vocos from huggingface charactr/vocos-mel-24khz

vocab :  /usr/local/lib/python3.13/dist-packages/f5_tts/infer/examples/vocab.txt
token :  custom
model :  /root/.cache/huggingface/hub/models--SWivid--F5-TTS/snapshots/84e5a410d9cead4de2f847e7c9369a6440bdfaca/F5TTS_v1_Base/model_1250000.safetensors 

Converting audio...
Using custom reference text...

ref_text   Welcome back. Stock prices are moving and investors are asking the same question. What happens next? Let's get started. 
gen_text 0 Did you know that even though Apple’s internal controls are audited, the report explicitly states they might not always prevent or detect errors?
gen_text 1 This is FinTok — your daily dose of financial knowledge.


Generating audio in 2 batches...


100%|██████████| 2/2 [00:09<00:00,  4.80s/it]

✓ F5-TTS terminado


## 9. Escuchar F5-TTS

In [55]:
display(Audio("/content/output_f5.wav"))

# MODELO 2 — XTTS-v2

XTTS-v2 puede clonar una voz directamente desde un audio de referencia y generar inglés sin necesitar la transcripción del clip.

> **Nota:** Coqui XTTS utiliza su propia licencia de modelo. Si el proyecto fuese comercial, revisa la licencia antes de usarlo.

## 10. Liberar memoria e instalar XTTS-v2

Liberamos F5-TTS antes de cargar el siguiente modelo para reducir el riesgo de quedarnos sin VRAM en una T4.

In [ ]:
import gc, torch

try:
    del f5
except:
    pass

gc.collect()
torch.cuda.empty_cache()

!pip -q install -U TTS

## 11. Generar audio con XTTS-v2

La primera vez descargará los pesos.

Si Coqui pregunta si aceptas la licencia del modelo, responde según corresponda para tu uso.

In [ ]:
import torch
from TTS.api import TTS

device = "cuda" if torch.cuda.is_available() else "cpu"

xtts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

xtts.tts_to_file(
    text=TARGET_TEXT,
    speaker_wav="/content/reference.wav",
    language="en",
    file_path="/content/output_xtts.wav",
)

print("✓ XTTS-v2 terminado")

## 12. Escuchar XTTS-v2

In [ ]:
display(Audio("/content/output_xtts.wav"))

# MODELO 3 — OpenVoice V2

OpenVoice funciona de forma algo distinta:

1. genera primero el texto con una voz inglesa base;
2. extrae el timbre de tu grabación;
3. transfiere ese timbre al audio generado.

Esto hace que sea una comparación interesante frente a F5-TTS y XTTS.

## 13. Liberar memoria e instalar OpenVoice V2 + MeloTTS

Esta instalación es algo más larga que las anteriores.

In [ ]:
import gc, torch

try:
    del xtts
except:
    pass

gc.collect()
torch.cuda.empty_cache()

!pip -q install -U unidic
!python -m unidic download
!pip -q install git+https://github.com/myshell-ai/MeloTTS.git
!git clone -q https://github.com/myshell-ai/OpenVoice.git /content/OpenVoice
!pip -q install -e /content/OpenVoice
!mkdir -p /content/checkpoints_v2
!wget -q -O /content/checkpoints_v2.zip https://myshell-public-repo-host.s3.amazonaws.com/openvoice/checkpoints_v2_0417.zip
!unzip -q -o /content/checkpoints_v2.zip -d /content/

## 14. Generar audio base y transferir tu timbre con OpenVoice V2

Se usa una voz inglesa base de MeloTTS y después OpenVoice aplica el timbre extraído de tu grabación.

El resultado final será `/content/output_openvoice.wav`.

In [ ]:
import os
import sys
import torch

sys.path.append("/content/OpenVoice")

from openvoice import se_extractor
from openvoice.api import ToneColorConverter
from melo.api import TTS as MeloTTS

device = "cuda:0" if torch.cuda.is_available() else "cpu"

converter = ToneColorConverter(
    "/content/checkpoints_v2/converter/config.json",
    device=device
)
converter.load_ckpt("/content/checkpoints_v2/converter/checkpoint.pth")

target_se, _ = se_extractor.get_se(
    "/content/reference.wav",
    converter,
    vad=True
)

melo = MeloTTS(language="EN", device=device)

speaker_ids = melo.hps.data.spk2id
print("Voces inglesas disponibles:", speaker_ids)

# Preferimos inglés americano si está disponible.
preferred = [
    "EN-US",
    "EN-Default",
    "EN-BR",
    "EN_INDIA",
    "EN-AU"
]

speaker_key = next(
    (name for name in preferred if name in speaker_ids),
    next(iter(speaker_ids))
)
speaker_id = speaker_ids[speaker_key]

base_path = "/content/openvoice_base.wav"

melo.tts_to_file(
    TARGET_TEXT,
    speaker_id,
    base_path,
    speed=1.0
)

# Embedding de la voz base.
source_se = torch.load(
    f"/content/checkpoints_v2/base_speakers/ses/{speaker_key.lower().replace('_', '-')}.pth",
    map_location=device
)

converter.convert(
    audio_src_path=base_path,
    src_se=source_se,
    tgt_se=target_se,
    output_path="/content/output_openvoice.wav",
    message="@MyShell"
)

print("✓ OpenVoice V2 terminado")

## 15. Escuchar OpenVoice V2

In [ ]:
display(Audio("/content/output_openvoice.wav"))

# 16. Comparación final

Escucha los tres con auriculares y valora principalmente:

- **Similitud de voz** — ¿suena realmente a ti?
- **Naturalidad** — ¿parece una persona real?
- **Prosodia** — ritmo, pausas, énfasis e entonación.
- **Pronunciación inglesa** — especialmente si tu audio de referencia tiene acento.
- **Artefactos** — sonidos metálicos, respiraciones extrañas, palabras cortadas.
- **Estabilidad** — si mantiene la misma voz durante todo el fragmento.

No des por ganador al que tenga simplemente la voz más bonita: para voice cloning interesa especialmente **similitud + naturalidad**.

In [ ]:
from IPython.display import Audio, display, Markdown
from pathlib import Path

outputs = {
    "1. F5-TTS": "/content/output_f5.wav",
    "2. XTTS-v2": "/content/output_xtts.wav",
    "3. OpenVoice V2": "/content/output_openvoice.wav",
}

for name, path in outputs.items():
    display(Markdown(f"### {name}"))
    if Path(path).exists():
        display(Audio(path))
    else:
        print("No se ha generado:", path)

## 17. Descargar los resultados

Ejecuta esta celda para crear un ZIP con:

- la grabación de referencia normalizada;
- F5-TTS;
- XTTS-v2;
- OpenVoice V2.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

folder = Path("/content/voice_comparison")
folder.mkdir(exist_ok=True)

for filename in [
    "reference.wav",
    "output_f5.wav",
    "output_xtts.wav",
    "output_openvoice.wav",
]:
    src = Path("/content") / filename
    if src.exists():
        shutil.copy(src, folder / filename)

zip_path = shutil.make_archive(
    "/content/voice_comparison_results",
    "zip",
    folder
)

print("ZIP creado:", zip_path)
files.download(zip_path)

# Qué haría después

Una vez escuchados los resultados:

- Si **F5-TTS** conserva mejor tu identidad y prosodia → úsalo como primera opción para generar el audio de tus vídeos.
- Si **XTTS-v2** clona especialmente bien tu voz → es muy cómodo para un pipeline sencillo de texto → voz.
- Si **OpenVoice V2** te da buen timbre pero suena demasiado a la voz base → probablemente F5/XTTS sean mejores para tu caso.

Para vídeos largos, no conviene generar varios minutos de una sola vez. Lo normal es dividir el guion por frases o párrafos, sintetizar cada segmento y después concatenarlos. Eso también permite controlar pausas y regenerar únicamente las frases que hayan salido mal.

### Problemas habituales

**CUDA out of memory**  
Ejecuta de nuevo la celda de liberación de memoria del modelo correspondiente. Si sigue ocurriendo, reinicia el entorno y ejecuta solo el modelo que quieras probar.

**La voz se parece poco**  
Usa una referencia limpia de 10–15 s, sin música, con voz natural y volumen estable.

**F5 pronuncia cosas raras**  
Comprueba que `REFERENCE_TEXT` sea exactamente lo que dices en `reference.wav`.

**El inglés tiene una entonación extraña**  
Prueba una referencia en la que hables inglés de forma natural. El objetivo es clonar tu voz, no necesariamente eliminar tu acento.

**Quiero cambiar el guion**  
Solo cambia `TARGET_TEXT` y vuelve a ejecutar la generación del modelo que quieras; no necesitas volver a subir la grabación.